# Titanic Survival Analysis

This notebook downloads a public Titanic CSV file and performs a typical survival prediction analysis.

## Goals

- Download the Titanic passenger dataset as a CSV file.
- Check missing values, basic statistics, and survival distribution.
- Explore relationships between survival and features such as sex, passenger class, age, fare, and family size.
- Build preprocessing pipelines.
- Compare Logistic Regression and Random Forest models.
- Evaluate the best model and inspect feature importance.

## Dataset

This notebook uses the public CSV file from DataScienceDojo.  
The file has the same structure as the classic Kaggle Titanic `train.csv`, and the target variable is `Survived`.

In [ ]:
# If required packages are not installed, uncomment and run the following line.
# %pip install pandas numpy matplotlib scikit-learn joblib

In [ ]:
from pathlib import Path
import re
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestClassifier
from sklearn.impute import SimpleImputer
from sklearn.inspection import permutation_importance
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    ConfusionMatrixDisplay,
    RocCurveDisplay,
    accuracy_score,
    classification_report,
    f1_score,
    precision_score,
    recall_score,
    roc_auc_score,
)
from sklearn.model_selection import StratifiedKFold, cross_validate, train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

warnings.filterwarnings("ignore")

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 120)

RANDOM_STATE = 42

## 1. Download and load the CSV file

When you run this notebook, the CSV file is saved as `data/titanic.csv`.  
If the file already exists, it is overwritten by the same public source.

In [ ]:
DATA_URL = "https://raw.githubusercontent.com/datasciencedojo/datasets/master/titanic.csv"

DATA_DIR = Path("data")
DATA_DIR.mkdir(exist_ok=True)

CSV_PATH = DATA_DIR / "titanic.csv"

df = pd.read_csv(DATA_URL)
df.to_csv(CSV_PATH, index=False)

print(f"Saved CSV to: {CSV_PATH.resolve()}")
print(f"Rows: {df.shape[0]:,}, Columns: {df.shape[1]:,}")

df.head()

## 2. Basic data inspection

In [ ]:
print("Shape:", df.shape)
display(df.head())
display(df.tail())

print("\nData types:")
display(df.dtypes.to_frame("dtype"))

print("\nBasic statistics:")
display(df.describe(include="all").T)

In [ ]:
missing = (
    df.isna()
    .sum()
    .to_frame("missing_count")
    .assign(missing_rate=lambda x: x["missing_count"] / len(df))
    .sort_values("missing_count", ascending=False)
)

display(missing)

ax = missing[missing["missing_count"] > 0]["missing_count"].plot(
    kind="bar",
    title="Missing values by column",
    figsize=(8, 4),
)
ax.set_xlabel("Column")
ax.set_ylabel("Missing count")
plt.tight_layout()
plt.show()

## 3. Distribution of the target variable: `Survived`

The `Survived` column means:

- `0`: The passenger did not survive.
- `1`: The passenger survived.

In [ ]:
target_counts = df["Survived"].value_counts().sort_index()
target_rate = df["Survived"].mean()

display(
    pd.DataFrame({
        "count": target_counts,
        "rate": target_counts / len(df),
    }).rename(index={0: "Died", 1: "Survived"})
)

print(f"Overall survival rate: {target_rate:.3f}")

ax = target_counts.rename(index={0: "Died", 1: "Survived"}).plot(
    kind="bar",
    title="Survival counts",
    figsize=(6, 4),
)
ax.set_xlabel("Outcome")
ax.set_ylabel("Count")
plt.tight_layout()
plt.show()

## 4. Survival rate by categorical variables

A typical Titanic analysis first checks survival rates by variables such as `Sex`, `Pclass`, and `Embarked`.

In [ ]:
def survival_rate_table(data: pd.DataFrame, column: str) -> pd.DataFrame:
    table = (
        data.groupby(column, dropna=False)
        .agg(
            passengers=("Survived", "size"),
            survived=("Survived", "sum"),
            survival_rate=("Survived", "mean"),
        )
        .sort_values("survival_rate", ascending=False)
    )
    return table

for column in ["Sex", "Pclass", "Embarked"]:
    print(f"\nSurvival rate by {column}")
    display(survival_rate_table(df, column))

In [ ]:
ax = df.groupby("Sex")["Survived"].mean().sort_values(ascending=False).plot(
    kind="bar",
    title="Survival rate by sex",
    figsize=(6, 4),
)
ax.set_xlabel("Sex")
ax.set_ylabel("Survival rate")
ax.set_ylim(0, 1)
plt.tight_layout()
plt.show()

In [ ]:
ax = df.groupby("Pclass")["Survived"].mean().sort_index().plot(
    kind="bar",
    title="Survival rate by passenger class",
    figsize=(6, 4),
)
ax.set_xlabel("Passenger class")
ax.set_ylabel("Survival rate")
ax.set_ylim(0, 1)
plt.tight_layout()
plt.show()

In [ ]:
ax = df.groupby("Embarked")["Survived"].mean().sort_values(ascending=False).plot(
    kind="bar",
    title="Survival rate by embarked port",
    figsize=(6, 4),
)
ax.set_xlabel("Embarked")
ax.set_ylabel("Survival rate")
ax.set_ylim(0, 1)
plt.tight_layout()
plt.show()

## 5. Numerical feature inspection

Here we inspect age, fare, and family-related numerical variables.

In [ ]:
numeric_columns = ["Age", "SibSp", "Parch", "Fare"]

display(df[numeric_columns + ["Survived"]].describe().T)

corr = df[numeric_columns + ["Survived"]].corr(numeric_only=True)
display(corr)

In [ ]:
ax = df["Age"].dropna().plot(
    kind="hist",
    bins=30,
    title="Age distribution",
    figsize=(7, 4),
)
ax.set_xlabel("Age")
ax.set_ylabel("Count")
plt.tight_layout()
plt.show()

In [ ]:
ax = df[df["Survived"] == 0]["Age"].dropna().plot(
    kind="hist",
    bins=30,
    alpha=0.5,
    label="Died",
    title="Age distribution by survival",
    figsize=(7, 4),
)
df[df["Survived"] == 1]["Age"].dropna().plot(
    kind="hist",
    bins=30,
    alpha=0.5,
    label="Survived",
    ax=ax,
)
ax.set_xlabel("Age")
ax.set_ylabel("Count")
ax.legend()
plt.tight_layout()
plt.show()

In [ ]:
ax = df["Fare"].plot(
    kind="hist",
    bins=40,
    title="Fare distribution",
    figsize=(7, 4),
)
ax.set_xlabel("Fare")
ax.set_ylabel("Count")
plt.tight_layout()
plt.show()

In [ ]:
ax = df.boxplot(
    column="Fare",
    by="Survived",
    figsize=(7, 4),
)
ax.set_title("Fare by survival")
ax.set_xlabel("Survived")
ax.set_ylabel("Fare")
plt.suptitle("")
plt.tight_layout()
plt.show()

## 6. Feature engineering

Typical Titanic analyses often create derived features such as:

- `Title`: Extracted from the passenger name, such as Mr, Mrs, Miss, or Master.
- `FamilySize`: `SibSp + Parch + 1`.
- `IsAlone`: Whether the passenger traveled alone.
- `CabinKnown`: Whether the cabin value is recorded.

In [ ]:
def extract_title(name: str) -> str:
    match = re.search(r",\s*([^\.]+)\.", name)
    if match:
        return match.group(1).strip()
    return "Unknown"

df_fe = df.copy()

df_fe["Title"] = df_fe["Name"].map(extract_title)

title_counts = df_fe["Title"].value_counts()
rare_titles = title_counts[title_counts < 10].index
df_fe["Title"] = df_fe["Title"].replace(rare_titles, "Rare")

df_fe["FamilySize"] = df_fe["SibSp"] + df_fe["Parch"] + 1
df_fe["IsAlone"] = (df_fe["FamilySize"] == 1).astype(int)
df_fe["CabinKnown"] = df_fe["Cabin"].notna().astype(int)

display(df_fe[["Name", "Title", "SibSp", "Parch", "FamilySize", "IsAlone", "Cabin", "CabinKnown"]].head(10))
display(survival_rate_table(df_fe, "Title"))
display(survival_rate_table(df_fe, "IsAlone"))
display(survival_rate_table(df_fe, "CabinKnown"))

In [ ]:
ax = df_fe.groupby("FamilySize")["Survived"].mean().plot(
    kind="bar",
    title="Survival rate by family size",
    figsize=(7, 4),
)
ax.set_xlabel("Family size")
ax.set_ylabel("Survival rate")
ax.set_ylim(0, 1)
plt.tight_layout()
plt.show()

## 7. Preprocessing pipeline for machine learning

We split the data into features and target.

- Numerical variables: Missing values are filled with the median, then standardized.
- Categorical variables: Missing values are filled with the most frequent value, then one-hot encoded.

Although `Pclass` is stored as a number, we treat it as a categorical variable.

In [ ]:
target = "Survived"

numeric_features = [
    "Age",
    "SibSp",
    "Parch",
    "Fare",
    "FamilySize",
    "IsAlone",
    "CabinKnown",
]

categorical_features = [
    "Pclass",
    "Sex",
    "Embarked",
    "Title",
]

feature_columns = numeric_features + categorical_features

X = df_fe[feature_columns]
y = df_fe[target]

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=RANDOM_STATE,
    stratify=y,
)

print("Train:", X_train.shape, "Test:", X_test.shape)
display(X_train.head())

In [ ]:
def make_one_hot_encoder():
    # The OneHotEncoder argument name changed across scikit-learn versions.
    # This helper supports both older and newer versions.
    try:
        return OneHotEncoder(handle_unknown="ignore", sparse_output=False)
    except TypeError:
        return OneHotEncoder(handle_unknown="ignore", sparse=False)

numeric_transformer = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler()),
    ]
)

categorical_transformer = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("onehot", make_one_hot_encoder()),
    ]
)

preprocess = ColumnTransformer(
    transformers=[
        ("num", numeric_transformer, numeric_features),
        ("cat", categorical_transformer, categorical_features),
    ]
)

## 8. Baseline model comparison

We compare two common baseline models.

1. Logistic Regression  
   - A simple and interpretable linear model.
2. Random Forest  
   - A tree-based model that can handle nonlinear relationships.

The models are evaluated with 5-fold cross validation.

In [ ]:
models = {
    "Logistic Regression": Pipeline(
        steps=[
            ("preprocess", preprocess),
            ("model", LogisticRegression(max_iter=1000, random_state=RANDOM_STATE)),
        ]
    ),
    "Random Forest": Pipeline(
        steps=[
            ("preprocess", preprocess),
            (
                "model",
                RandomForestClassifier(
                    n_estimators=300,
                    min_samples_leaf=2,
                    random_state=RANDOM_STATE,
                    n_jobs=-1,
                ),
            ),
        ]
    ),
}

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)

scoring = {
    "accuracy": "accuracy",
    "precision": "precision",
    "recall": "recall",
    "f1": "f1",
    "roc_auc": "roc_auc",
}

rows = []

for name, model in models.items():
    scores = cross_validate(
        model,
        X_train,
        y_train,
        cv=cv,
        scoring=scoring,
        n_jobs=-1,
        return_train_score=False,
    )

    row = {"model": name}
    for metric in scoring:
        row[f"{metric}_mean"] = scores[f"test_{metric}"].mean()
        row[f"{metric}_std"] = scores[f"test_{metric}"].std()
    rows.append(row)

cv_results = pd.DataFrame(rows).sort_values("roc_auc_mean", ascending=False)
display(cv_results)

In [ ]:
ax = cv_results.set_index("model")["roc_auc_mean"].plot(
    kind="bar",
    title="Cross-validation ROC-AUC by model",
    figsize=(7, 4),
)
ax.set_xlabel("Model")
ax.set_ylabel("Mean ROC-AUC")
ax.set_ylim(0, 1)
plt.tight_layout()
plt.show()

## 9. Final evaluation on the test set

We select the model with the highest cross-validation ROC-AUC and evaluate it on the held-out test set.

In [ ]:
best_name = cv_results.iloc[0]["model"]
best_model = models[best_name]

best_model.fit(X_train, y_train)

y_pred = best_model.predict(X_test)
y_proba = best_model.predict_proba(X_test)[:, 1]

test_metrics = pd.DataFrame(
    {
        "metric": ["accuracy", "precision", "recall", "f1", "roc_auc"],
        "value": [
            accuracy_score(y_test, y_pred),
            precision_score(y_test, y_pred),
            recall_score(y_test, y_pred),
            f1_score(y_test, y_pred),
            roc_auc_score(y_test, y_proba),
        ],
    }
)

print(f"Best model: {best_name}")
display(test_metrics)

print("\nClassification report:")
print(classification_report(y_test, y_pred, target_names=["Died", "Survived"]))

In [ ]:
ConfusionMatrixDisplay.from_predictions(
    y_test,
    y_pred,
    display_labels=["Died", "Survived"],
)
plt.title(f"Confusion matrix: {best_name}")
plt.tight_layout()
plt.show()

In [ ]:
RocCurveDisplay.from_predictions(y_test, y_proba)
plt.title(f"ROC curve: {best_name}")
plt.tight_layout()
plt.show()

## 10. Feature importance

Here we use `permutation_importance`, which is less tied to a specific model type than built-in tree importance.

It measures how much the ROC-AUC score decreases when one feature is randomly shuffled.

In [ ]:
perm = permutation_importance(
    best_model,
    X_test,
    y_test,
    n_repeats=20,
    random_state=RANDOM_STATE,
    scoring="roc_auc",
    n_jobs=-1,
)

importance = (
    pd.DataFrame(
        {
            "feature": X_test.columns,
            "importance_mean": perm.importances_mean,
            "importance_std": perm.importances_std,
        }
    )
    .sort_values("importance_mean", ascending=False)
    .reset_index(drop=True)
)

display(importance)

top_n = min(10, len(importance))
ax = importance.head(top_n).set_index("feature")["importance_mean"].sort_values().plot(
    kind="barh",
    title=f"Top {top_n} permutation importances",
    figsize=(7, 5),
)
ax.set_xlabel("Mean decrease in ROC-AUC")
ax.set_ylabel("Feature")
plt.tight_layout()
plt.show()

## 11. Typical interpretation

After running this notebook, you will usually observe patterns such as:

- `Sex` is often a very strong feature, and female passengers usually have a higher survival rate.
- `Pclass` is also important; higher passenger classes tend to have higher survival rates.
- `Age` contains missing values, so imputation is necessary before modeling.
- `Fare` has a skewed distribution, so outliers should be considered.
- Derived features such as `Title`, `FamilySize`, `IsAlone`, and `CabinKnown` may improve a simple baseline model.

However, this is not a causal analysis.  
This notebook is mainly a practice workflow for basic machine learning with the Titanic dataset.

## 12. Possible next improvements

- Impute `Age` using median values grouped by sex, title, and passenger class.
- Transform `Fare` with `np.log1p(Fare)`.
- Create additional features from `Ticket` and `Cabin`.
- Tune hyperparameters with GridSearchCV or RandomizedSearchCV.
- Load Kaggle's `test.csv` and create a submission CSV.